# 📘 Decision Tree – Study Notes
---

## 1. What is a Decision Tree?

- A versatile algorithm that works for both **classification** and **regression**
- Divides the dataset into a **tree-like structure** based on conditions/rules
- Makes predictions based on those conditions
- Easy to visualize and understand — even for non-technical people
- Works well with complex datasets
- Performs even better when combined with **ensemble methods** (Random Forest, Boosting)

**How it works (simple example — should I go out on Friday night?):**
- Root node: Is it past 10 PM?
- If Yes → Stay home
- If No → Are friends available? → Go out or Stay home
- Tree depth = number of levels of questions asked

## 2. Decision Tree for Regression

- Divides X values into **non-overlapping regions** R1, R2, ..., RJ
- For any observation falling in region Rj → prediction = **mean of Y values in that region**
- Regions are chosen to **minimize RSS** (Residual Sum of Squares)

### Recursive Binary Splitting (Greedy Approach)
- Splits nodes into **two branches** at each step
- **Greedy** = picks the best split right now, not the globally best split
- Finds a threshold `s` for each feature such that splitting at `s` gives **minimum RSS**
- Continues splitting until a stopping condition is met
- ⚠️ High chance of **overfitting** without pruning

## 3. Tree Pruning (Controlling Overfitting)

Like regularization in linear regression, pruning trims the tree to reduce complexity.

**Formula:**
$$\text{Cost} = \sum RSS + \alpha |T|$$

- `T` = subtree (subset of full tree)
- `α` = tuning parameter (penalizes tree size)
- Best `α` chosen via **cross-validation**

### Pre-pruning (Forward Pruning)
- Stops splitting **before** the tree fully grows
- Uses conditions to decide when to stop early

### Post-pruning (Backward Pruning)
- Grows the **full tree first**, then removes non-significant branches
- Uses cross-validation to check if removing a branch improves or reduces accuracy
- If removing a branch reduces accuracy → keep it; else → convert to leaf node

## 4. Decision Tree for Classification

- Instead of RSS, splits are made using:
  - **Entropy** + Information Gain
  - **Gini Impurity**

### Entropy
- Measures **randomness / impurity** in data
- Low entropy = clean node (one class dominates)
- High entropy = mixed node (classes are balanced)

$$E = -p \log_2(p) - q \log_2(q)$$

- `p` = fraction of class 1, `q` = 1 - p (fraction of class 2)
- E = 0 → all same class (pure) ✅
- E = 1 → 50/50 split (most impure) ❌

### Information Gain
- Measures **decrease in entropy** after a split
- Higher information gain = better split
$$\text{Information Gain} = \text{Entropy(before)} - \text{Entropy(after split)}$$

### Gini Impurity
- Measures how often a random element would be **incorrectly classified**
- Range: 0 (pure) to 1 (completely random)
- The node with **lowest Gini** is selected as the split node
- Formula: multiply probability of each class being correct × sum of all incorrect probabilities

### Gini vs Entropy
- Both give **very similar results** in practice
- Gini is **faster to compute** (no log calculation)
- That's why **CART uses Gini by default**

## 5. How Root Node is Selected (Worked Example Logic)

Given features (e.g., Class and Gender):
1. Calculate Gini impurity for **every possible split** of each feature
2. Pick the feature + split that gives **lowest weighted Gini impurity**
3. That becomes the **root node**
4. Repeat the same process for child nodes

Same logic applies using Entropy + Information Gain:
1. Calculate entropy before split
2. Calculate entropy after each possible split
3. Pick split with **highest Information Gain**

## 6. Decision Tree Algorithms

| Algorithm | Criteria | Handles |
|---|---|---|
| **ID3** | Information Gain (Entropy) | Categorical only |
| **C4.5** | Information Gain (improved) | Categorical + Continuous |
| **CART** | Gini Impurity (default) | Classification + Regression |

- **sklearn uses CART** by default
- CART can also use `entropy` if specified via `criterion` parameter

## 7. Advantages & Disadvantages

### ✅ Advantages
- Works for both **regression and classification**
- Very easy to **visualize and interpret**
- **No need for scaling or normalization**
- Handles complex datasets well

### ❌ Disadvantages
- **High chance of overfitting** (especially without pruning)
- Small change in data → big change in tree structure (unstable)
- **Slower to train** than other classifiers
- Greedy approach → not guaranteed to find the best overall tree

## 8. Cross-Validation

**Problem:** Training and testing on the same data = overfitting, gives falsely high accuracy.

**Solution:** Cross-validation — split data into train and test, evaluate on unseen data.

### Hold-Out Method
- Simply split data into train and test (e.g., 70/30)
- Fast but **high variance** — result depends heavily on which data ends up in train/test

### k-Fold Cross-Validation
- Split data into `k` equal parts
- Train on `k-1` parts, test on the remaining 1 part
- Repeat `k` times, each time using a different part as the test set
- Final CV error = **mean of all k errors**
- More reliable than hold-out; more computationally expensive
- Common values: k = 5 or k = 10

### LOOCV (Leave One Out Cross-Validation)
- Special case of k-fold where `k = n` (number of observations)
- Each time, one single observation is the test set
- Very low bias but **high variance and very slow**

### Which to use?
| Method | Bias | Variance | Speed |
|---|---|---|---|
| Hold-Out | High | High | Fast |
| k-Fold (k=5 or 10) | Medium | Low | Medium |
| LOOCV | Low | High | Very Slow |

> **Best practical choice: k-Fold with k=5 or k=10** — good balance of bias, variance, and speed

## 9. Hyperparameters of Decision Tree (sklearn)

| Parameter | What it does | Default |
|---|---|---|
| `criterion` | Split quality measure: `'gini'` or `'entropy'` | `'gini'` |
| `splitter` | How to split: `'best'` or `'random'` | `'best'` |
| `max_depth` | Max depth of tree (None = grow fully) | None |
| `min_samples_split` | Min samples needed to split a node | 2 |
| `min_samples_leaf` | Min samples required at a leaf node | 1 |
| `max_features` | Max features to consider for best split | None |
| `max_leaf_nodes` | Max number of leaf nodes allowed | None |
| `min_impurity_decrease` | Split only if impurity decreases by this amount | 0 |
| `class_weight` | Handle imbalanced classes | None |
| `random_state` | Seed for reproducibility | None |

## 10. GridSearchCV – Hyperparameter Tuning

- **Goal:** Find the best combination of hyperparameters automatically
- Tries **all combinations** of given parameter values (exhaustive search)
- Uses **cross-validation** score to pick the best combination

```python
from sklearn.model_selection import GridSearchCV

grid_param = {
    'criterion': ['gini', 'entropy'],
    'max_depth': range(2, 32, 1),
    'min_samples_leaf': range(1, 10, 1),
    'min_samples_split': range(2, 10, 1),
    'splitter': ['best', 'random']
}

grid_search = GridSearchCV(estimator=clf, param_grid=grid_param, cv=5, n_jobs=-1)
grid_search.fit(x_train, y_train)

print(grid_search.best_params_)   # best combination
print(grid_search.best_score_)    # best CV score
```

> ⚠️ GridSearchCV doesn't always guarantee the perfect result — it depends on which parameters you pass in. Hit and trial is also needed.

## 11. PCA for Feature Selection

- **PCA (Principal Component Analysis)** reduces the number of features while keeping most of the variance
- Used when there are many correlated or redundant features
- Example from notebook: 11 features → 8 principal components explained ~95% of variance

```python
from sklearn.decomposition import PCA
pca = PCA(n_components=8)
new_data = pca.fit_transform(x_scaled)
```

- Plot cumulative explained variance to decide how many components to keep

## 12. Reading the Decision Tree Output

Each node in the visualized tree shows:
- **Feature & condition** used for the split (e.g., `alcohol <= 10.5`)
- **Gini impurity** of that node
- **Samples** = number of training observations at that node
- **Value** = count of observations per class (e.g., `[8, 38, 468, ...]`)

Use `export_graphviz` + `pydotplus` to visualize the tree.

## 13. Practical Implementation – Wine Quality Dataset

**Steps followed in the notebook:**

1. **Load data** – `winequality_red.csv` (11 features → predict wine quality 0–10)
2. **Train/test split** – 70% train, 30% test
3. **Fit basic tree** – No preprocessing, no tuning → baseline accuracy
4. **Scale data** – `StandardScaler`
5. **PCA** – Reduce 11 features to 8 (95% variance retained)
6. **GridSearchCV** – Tune criterion, max_depth, min_samples_leaf, etc.
7. **Refit with best params** → improved test accuracy
8. **Visualize tree** – Using `export_graphviz` + `pydotplus`
9. **Save model** – Using `pickle` (saves classifier, scaler, and PCA separately)

```python
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(criterion='entropy', max_depth=24,
                              min_samples_leaf=1, min_samples_split=2,
                              splitter='random')
clf.fit(x_train, y_train)
clf.score(x_test, y_test)
```

## 14. Quick Summary

| Concept | Key Point |
|---|---|
| Decision Tree | Works for classification and regression |
| Greedy Approach | Best split at current step, not globally |
| Entropy | Measures randomness; 0 = pure, 1 = mixed |
| Information Gain | Entropy before − Entropy after split |
| Gini Impurity | Probability of misclassification; 0 = pure |
| CART | Default sklearn algorithm; uses Gini |
| Pruning | Prevents overfitting by trimming the tree |
| Pre-pruning | Stop growing early |
| Post-pruning | Grow full tree, then trim |
| Hold-Out CV | Fast, high variance |
| k-Fold CV | Best balance of bias/variance; k=5 or 10 |
| LOOCV | Low bias, high variance, very slow |
| GridSearchCV | Exhaustively finds best hyperparameters |
| PCA | Reduces features while keeping variance |
| No scaling needed | Decision trees don't require normalization |